# Imidazolone & Thiazolone Library Generation - First Phase

Virtual combinatorial library of COX-2 selective inhibitors derived from  
commercially available aldehydes, carboxylic acids and primary amines via  
reactions *in silico*.

**Pipeline:** ```Aldehydes + Carboxylic Acids → Erlenmeyer-Plöchl → Oxazolones → Aminolysis-GFPc → Imidazolones```  

**Amines** feed into the Aminolysis-GFPc step as co-reactants.  

**Parallel branch:** ```Oxazolones → Sulphur-Exchange → Thiazolones```  

All three building-block sets pass Price → Veber filters before reactions.  
A hard global cutoff (`PriceMol <= 200.00`) is enforced in Veber filters and reaction pairing.  
Final products (Imidazolones and Thiazolones) are filtered by Veber and  
Brenk+PAINS before export, and saved as two **separate** CSV files.

***(Please read [the first phase README.md](01_library_mols_data/README.md) for full details on this notebook).***

In [1]:
import gc
import sys
import pandas as pd

sys.path.insert(0, "01_library_mols_data")

from modules.io import sdf_to_dataframe, report_df_size, add_rdkit_properties, save_dataframe_as_csv
from modules.filters import filter_Veber, filter_BrenkPAINS
from modules.reactions import rxn_ErlenmeyerPlochl, rxn_AminolysisGFPc, rxn_SulphurExchange
from modules.enamine_api import EnamineClient, add_enamine_prices
from modules.pipeline import CheckpointManager, stage_path, rejected_path, load_or_run, load_or_filter, save_dataframe, init_stage_dirs

init_stage_dirs()

FORCE_RECOMPUTE = False
MAX_PRICE_MOL = 200.00  # Global cutoff for PriceMol; set to None to disable filtering

__version__ = "5.5"
print(f"library v{__version__}")
print("Checkpoint system enabled")
print(f"Global PriceMol cutoff: <= {MAX_PRICE_MOL}")

library v5.5
Checkpoint system enabled
Global PriceMol cutoff: <= 200.0


## 📥 1. Load SDFs

The SDFs of the compounds provided by Enamine (with their IDs) that match a
specific substructure are loaded:
- Aldehydes, R-CHO
- Carboxylic compounds, R-COOH or RCOX
- Primary amines, R-NH₂  
  
  
> SDF loading is fast — no caching needed.

In [2]:
# === CHOOSE DATASET ===
# Uncomment ONE of the two options below:

SDF_PATHS = {      # OPTION 1: Full SDF dataset (use for production runs)
    "aldehydes":   ".raw_data_curation/Enamine Full SDFs/EnamineBBStock_site_Aldehydes_12040.sdf",
    "carboxylics": ".raw_data_curation/Enamine Full SDFs/EnamineBBStock_site_CarboxylicAcids_75732.sdf",
    "amines":      ".raw_data_curation/Enamine Full SDFs/EnamineBBStock_site_PrimaryAmines_68264.sdf",
}

# SDF_PATHS = {      # OPTION 2: Current Lab dataset (Purchased reagents only)
#     "aldehydes":   "01_library_mols_data/inputs/Purchased_Aldehydes_104.sdf",
#     "carboxylics": "01_library_mols_data/inputs/Purchased_CarboxylicAcids_45.sdf",
#     "amines":      "01_library_mols_data/inputs/Purchased_PrimaryAmines_62.sdf",
# }

# SDF_PATHS = {      # OPTION 3: TESTER dataset (use for testing/validation)
#     "aldehydes":   "01_library_mols_data/inputs/TESTER_Aldehydes_120.sdf",
#     "carboxylics": "01_library_mols_data/inputs/TESTER_CarboxylicAcids_750.sdf",
#     "amines":      "01_library_mols_data/inputs/TESTER_PrimaryAmines_680.sdf",
# }

df_aldehydes_raw   = sdf_to_dataframe(SDF_PATHS["aldehydes"],   id_prefix="A") 
df_carboxylics_raw = sdf_to_dataframe(SDF_PATHS["carboxylics"], id_prefix="C")
df_amines_raw      = sdf_to_dataframe(SDF_PATHS["amines"],      id_prefix="N")

[SDF Reader] Loaded 12040 unique compounds from .raw_data_curation/Enamine Full SDFs/EnamineBBStock_site_Aldehydes_12040.sdf
[SDF Reader] Loaded 75732 unique compounds from .raw_data_curation/Enamine Full SDFs/EnamineBBStock_site_CarboxylicAcids_75732.sdf
[SDF Reader] Loaded 68264 unique compounds from .raw_data_curation/Enamine Full SDFs/EnamineBBStock_site_PrimaryAmines_68264.sdf


## 🔸 2. Price Filtration

Query Enamine Store API for real-time pricing. Compounds exceeding price thresholds are discarded during retrieval.

> Internal caching: prices are cached in `01_library_mols_data/.interim/building_blocks/.cache/price_cache/`.

In [3]:
client = EnamineClient()

print("=== Aldehydes ===")
df_aldehydes_cheap = add_enamine_prices(df_aldehydes_raw, client=client)

print("\n=== Carboxylic Acids ===")
df_carboxylics_cheap = add_enamine_prices(df_carboxylics_raw, client=client)

print("\n=== Amines ===")
df_amines_cheap = add_enamine_prices(df_amines_raw, client=client)

report_df_size(df_aldehydes_cheap,   "Aldehydes - Cheap")
report_df_size(df_carboxylics_cheap, "Carboxylics - Cheap")
report_df_size(df_amines_cheap,      "Amines - Cheap")

=== Aldehydes ===
[Enamine Pricing] Processing 12040 compounds...
[Enamine Pricing] Querying 12040 compounds via API
[EnamineClient] Signed in successfully.
[Enamine Pricing] Completed: 1574/12040 with valid pricing
⚠️  Removed 10466 compounds (no valid pricing)

=== Carboxylic Acids ===
[Enamine Pricing] Loaded 1575 valid + 10465 invalid cached
[Enamine Pricing] Processing 75732 compounds...
[Enamine Pricing] Querying 75732 compounds via API
⚠️ Request error (attempt 1/3): 504 Server Error: Gateway Timeout for url: https://enaminestore.com:443/api/v2/catalog/search/by/codes
[Enamine Pricing] Completed: 4973/75732 with valid pricing
⚠️  Removed 70759 compounds (no valid pricing)

=== Amines ===
[Enamine Pricing] Loaded 6666 valid + 81106 invalid cached
[Enamine Pricing] Processing 68264 compounds...
[Enamine Pricing] Querying 68264 compounds via API
[Enamine Pricing] Completed: 5368/68264 with valid pricing
⚠️  Removed 62896 compounds (no valid pricing)
[Aldehydes - Cheap] 1574 rows
[C

## 🔸 3. Veber filtering (of building blocks)

Compute RDKit molecular descriptors, then apply per-class bioavailability criteria.  

Each building block class uses independent limits (`VEBER_ALDEHYDES`, `VEBER_CARBOXYLICS`,`VEBER_AMINES`) back-calculated  
from the final product thresholds using each reaction's property increments (ΔtPSA, ΔRotB, ΔLogP, ΔMW, ΔHBD, ΔHBA).  

| Reaction | ΔtPSA | ΔRotB | ΔLogP | ΔMW | ΔHBD | ΔHBA |
|----------|------|-------|--------|------|------|-----|
| E-P      | -15.71 | 0 | 0.4127 | +21.022 | -1 | +1 |
| A-G      | -32.01 | 1 | 0.3403 | -18.015 | -1 | -2 |
| S-E      | +16.07 | 0 | +0.7166 | +16.068 | -1 | 0 |

In [4]:
VEBER_BB = {
    "aldehydes":   dict(max_tPSA=118.41, max_RotB=10, max_LogP=4.4964, max_MWT=405.883, max_HBD=5, max_HBA=9),
    "carboxylics": dict(max_tPSA=138.64, max_RotB=10, max_LogP=4.3821, max_MWT=421.882, max_HBD=6, max_HBA=9),
    "amines":      dict(max_tPSA=133.35, max_RotB=9,  max_LogP=3.7943, max_MWT=377.873, max_HBD=6, max_HBA=9),
}

BB_ID_COLS = ["ID", "Catalog_ID", "SMILES", "PriceMol", "PriceG", "tPSA", "RotB", "LogP", "MW", "HBD", "HBA"]
BB_REJ_COLS = BB_ID_COLS + ["Rejection"]
PRODUCT_RAW_COLS = ["ID", "SMILES", "PriceMol"]

def _veber_bb_filter(df: pd.DataFrame, veber_params: dict) -> tuple[pd.DataFrame, pd.DataFrame]:
    return filter_Veber(
        add_rdkit_properties(df),
        max_price_mol=MAX_PRICE_MOL,
        **veber_params,
    )

df_aldehydes_druglike, _ = load_or_filter(
    lambda: _veber_bb_filter(df_aldehydes_cheap, VEBER_BB["aldehydes"]),
    accepted_csv=stage_path("Aldehydes", filter_mode="veber"),
    rejected_csv=rejected_path("Aldehydes", "veber"),
    force_recompute=FORCE_RECOMPUTE,
    input_row_count=len(df_aldehydes_cheap),
    params={"max_price_mol": MAX_PRICE_MOL, "veber": VEBER_BB["aldehydes"]},
)
report_df_size(df_aldehydes_druglike, "Aldehydes - Druglike")

df_carboxylics_druglike, _ = load_or_filter(
    lambda: _veber_bb_filter(df_carboxylics_cheap, VEBER_BB["carboxylics"]),
    accepted_csv=stage_path("Carboxylics", filter_mode="veber"),
    rejected_csv=rejected_path("Carboxylics", "veber"),
    force_recompute=FORCE_RECOMPUTE,
    input_row_count=len(df_carboxylics_cheap),
    params={"max_price_mol": MAX_PRICE_MOL, "veber": VEBER_BB["carboxylics"]},
)
report_df_size(df_carboxylics_druglike, "Carboxylics - Druglike")

df_amines_druglike, _ = load_or_filter(
    lambda: _veber_bb_filter(df_amines_cheap, VEBER_BB["amines"]),
    accepted_csv=stage_path("Amines", filter_mode="veber"),
    rejected_csv=rejected_path("Amines", "veber"),
    force_recompute=FORCE_RECOMPUTE,
    input_row_count=len(df_amines_cheap),
    params={"max_price_mol": MAX_PRICE_MOL, "veber": VEBER_BB["amines"]},
)
report_df_size(df_amines_druglike, "Amines - Druglike")

[load_or_filter] No cache pair matches input row count (1,574); recomputing
[load_or_filter] Computing Aldehydes_veber.csv...
[filter_Veber] 278/1,574 accepted (17.7%), 1,296 rejected
[load_or_filter] Saved Aldehydes_veber_278cmpds.csv (278 accepted) + Aldehydes_rejected_veber_1296cmpds.csv (1,296 rejected)
[Aldehydes - Druglike] 278 rows
[load_or_filter] No cache pair matches input row count (4,973); recomputing
[load_or_filter] Computing Carboxylics_veber.csv...
[filter_Veber] 744/4,973 accepted (15.0%), 4,229 rejected
[load_or_filter] Saved Carboxylics_veber_744cmpds.csv (744 accepted) + Carboxylics_rejected_veber_4229cmpds.csv (4,229 rejected)
[Carboxylics - Druglike] 744 rows
[load_or_filter] No cache pair matches input row count (5,368); recomputing
[load_or_filter] Computing Amines_veber.csv...
[filter_Veber] 836/5,368 accepted (15.6%), 4,532 rejected
[load_or_filter] Saved Amines_veber_836cmpds.csv (836 accepted) + Amines_rejected_veber_4532cmpds.csv (4,532 rejected)
[Amines - 

## 🔹 4. Erlenmeyer-Plöchl Reaction

Aldehydes + Carboxylic Acids → Oxazolones

In [5]:
# E-P reaction outputs to {Stage}_raw.csv (not {Stage}.csv)
df_oxazolones_raw = load_or_run(
    lambda checkpoint_manager: rxn_ErlenmeyerPlochl(
        df_aldehydes_druglike, 
        df_carboxylics_druglike,
        chunk_size=25,
        checkpoint_manager=checkpoint_manager,
        max_price_mol=MAX_PRICE_MOL,
    ),
    output_csv=stage_path("Oxazolones", filter_mode="raw"),
    stage_name="Oxazolones",
    force_recompute=FORCE_RECOMPUTE,
    params={"max_price_mol": MAX_PRICE_MOL, "chunk_size": 25},
)

report_df_size(df_oxazolones_raw, "Oxazolones - Raw")

[load_or_run] Resuming from checkpoint: 0/0 chunks completed
[ErlenmeyerPlochl] Removed descriptor columns before reaction: Atoms, CAtm, HBA, HBD, HetAtm, HvyAtm ... (+6 more)
[ErlenmeyerPlochl] Using minimal reaction columns ['SMILES', 'ID', 'PriceMol'] (dropped 14 non-reactive columns)
[ErlenmeyerPlochl] Removed descriptor columns before reaction: Atoms, CAtm, HBA, HBD, HetAtm, HvyAtm ... (+6 more)
[ErlenmeyerPlochl] Using minimal reaction columns ['SMILES', 'ID', 'PriceMol'] (dropped 14 non-reactive columns)
[ErlenmeyerPlochl] Processing chunk 1/12 (25 aldehydes × 744 carboxylics)
[ErlenmeyerPlochl] 206,832 total pairs (12 chunks of ~25 aldehydes)
[ErlenmeyerPlochl] Processing chunk 2/12 (25 aldehydes × 744 carboxylics)
[ErlenmeyerPlochl] Processing chunk 4/12 (25 aldehydes × 744 carboxylics)
[ErlenmeyerPlochl] Processing chunk 6/12 (25 aldehydes × 744 carboxylics)
[ErlenmeyerPlochl] Processing chunk 8/12 (25 aldehydes × 744 carboxylics)
[ErlenmeyerPlochl] Processing chunk 10/12 (25

## 🔸 5. Veber Filter (Oxazolones)

Apply bioavailability criteria to oxazolone intermediates.
Limits are set to the most permissive value across both downstream pathways (A-G and S-E)
to avoid discarding valid precursors prematurely.

In [6]:
VEBER_OXAZOLONES = dict(
    max_tPSA=145.99, max_RotB=10, max_LogP=5.0848,
    max_MWT=486.957, max_HBD=5, max_HBA=11
)

# Filter output: {Stage}_{N}cmpds.csv (row count in filename)
df_oxazolones_druglike, _ = load_or_filter(
    lambda: filter_Veber(
        add_rdkit_properties(df_oxazolones_raw),
        max_price_mol=MAX_PRICE_MOL,
        **VEBER_OXAZOLONES,
    ),
    accepted_csv=stage_path("Oxazolones", filter_mode="veber"),
    rejected_csv=rejected_path("Oxazolones", "veber"),
    force_recompute=FORCE_RECOMPUTE,
    input_row_count=len(df_oxazolones_raw),
    params={"max_price_mol": MAX_PRICE_MOL, "veber": VEBER_OXAZOLONES},
)

report_df_size(df_oxazolones_druglike, "Oxazolones - Druglike")

[load_or_filter] No cache pair matches input row count (80,557); recomputing
[load_or_filter] Computing Oxazolones_veber.csv...
[filter_Veber] 72,220/80,557 accepted (89.7%), 8,337 rejected
[load_or_filter] Saved Oxazolones_veber_72220cmpds.csv (72,220 accepted) + Oxazolones_rejected_veber_8337cmpds.csv (8,337 rejected)
[Oxazolones - Druglike] 72220 rows


## 🔹 6. Aminolysis-GFPc Reaction

Oxazolones + Amines → Imidazolones

> ⚠️ This reaction generates billions of products. Checkpointing is enabled by default.

In [7]:
# Free memory — DataFrames no longer needed after E-P and Veber
del df_oxazolones_raw
del df_aldehydes_raw, df_carboxylics_raw, df_amines_raw
del df_aldehydes_cheap, df_carboxylics_cheap, df_amines_cheap
del df_aldehydes_druglike, df_carboxylics_druglike
gc.collect()
print("[Memory] Freed unnecessary DataFrames before A-G")

# A-G reaction outputs to {Stage}_raw.csv
df_imidazolones_raw = load_or_run(
    lambda checkpoint_manager: rxn_AminolysisGFPc(
        df_oxazolones_druglike, 
        df_amines_druglike,
        chunk_size=10,
        checkpoint_manager=checkpoint_manager,
        max_price_mol=MAX_PRICE_MOL,
    ),
    output_csv=stage_path("Imidazolones", filter_mode="raw"),
    stage_name="Imidazolones",
    force_recompute=FORCE_RECOMPUTE,
    params={"max_price_mol": MAX_PRICE_MOL, "chunk_size": 10},
)

df_imidazolones_raw = df_imidazolones_raw[PRODUCT_RAW_COLS].copy()

report_df_size(df_imidazolones_raw, "Imidazolones - Raw")

[Memory] Freed unnecessary DataFrames before A-G
[load_or_run] Resuming from checkpoint: 0/0 chunks completed
[AminolysisGFPc] Removed descriptor columns before reaction: Atoms, CAtm, HBA, HBD, HetAtm, HvyAtm ... (+6 more)
[AminolysisGFPc] Using minimal reaction columns ['SMILES', 'ID', 'PriceMol'] (dropped 12 non-reactive columns)
[AminolysisGFPc] Removed descriptor columns before reaction: Atoms, CAtm, HBA, HBD, HetAtm, HvyAtm ... (+6 more)
[AminolysisGFPc] Using minimal reaction columns ['SMILES', 'ID', 'PriceMol'] (dropped 14 non-reactive columns)
[AminolysisGFPc] Pairing preflight: total=60,375,920, affordable=3,964,536, price_cutoff=200.0
[AminolysisGFPc] Processing chunk 500/7222 (4,990 oxazolones processed)
[AminolysisGFPc] Processing chunk 1000/7222 (9,990 oxazolones processed)
[AminolysisGFPc] Processing chunk 1500/7222 (14,990 oxazolones processed)
[AminolysisGFPc] Processing chunk 2000/7222 (19,990 oxazolones processed)
[AminolysisGFPc] Processing chunk 2500/7222 (24,990 ox

## 🔹 7. Sulphur-Exchange Reaction

Oxazolones → Thiazolones  

> **NB:** input is `df_oxazolones_druglike` (post-Veber, Cell 5) — not `df_imidazolones_raw` or `df_oxazolones_raw`.

In [8]:
THIOACETIC_PRICE_EQ = 10.0

# S-E reaction outputs to {Stage}_raw.csv
df_thiazolones_raw = load_or_run(
    lambda checkpoint_manager: rxn_SulphurExchange(
        df_oxazolones_druglike, 
        thioacetic_price_eq=THIOACETIC_PRICE_EQ,
        chunk_size=10000,
        checkpoint_manager=checkpoint_manager,
        max_price_mol=MAX_PRICE_MOL,
    ),
    output_csv=stage_path("Thiazolones", filter_mode="raw"),
    stage_name="Thiazolones",
    force_recompute=FORCE_RECOMPUTE,
    params={"max_price_mol": MAX_PRICE_MOL, "chunk_size": 10000, "thioacetic_price_eq": THIOACETIC_PRICE_EQ},
)

df_thiazolones_raw = df_thiazolones_raw[PRODUCT_RAW_COLS].copy()

report_df_size(df_thiazolones_raw, "Thiazolones - Raw")

[load_or_run] Resuming from checkpoint: 0/0 chunks completed
[SulphurExchange] Removed descriptor columns before reaction: Atoms, CAtm, HBA, HBD, HetAtm, HvyAtm ... (+6 more)
[SulphurExchange] Using minimal reaction columns ['SMILES', 'ID', 'PriceMol'] (dropped 12 non-reactive columns)
[SulphurExchange] Pairing preflight: total=72,220, affordable=61,299, price_cutoff=200.0
[SulphurExchange] 72,220 valid oxazolones: 0 cache hits, 61,299 misses
[SulphurExchange] Processing chunk 1/7 (10,000 items)
[SulphurExchange] Processing chunk 2/7 (10,000 items)
[SulphurExchange] Processing chunk 3/7 (10,000 items)
[SulphurExchange] Processing chunk 4/7 (10,000 items)
[SulphurExchange] Processing chunk 5/7 (10,000 items)
[SulphurExchange] Processing chunk 6/7 (10,000 items)
[SulphurExchange] Processing chunk 7/7 (1,299 items)
[SulphurExchange] Checkpoint saved: Thiazolones_checkpoint.json
[SulphurExchange] out=61,299 rows | inp=72,220 | ⚠️ skipped=10,921
[load_or_run] Saved Thiazolones_raw_61299cmpd

## 🔸 8. Veber Filter (Products)

Drop any stale descriptor columns, recompute product descriptors, and apply Veber criteria to both product families.  

`VEBER_PRODUCTS` defines the final target thresholds shared by imidazolones and thiazolones.

In [9]:
VEBER_PRODUCTS = dict(
    max_tPSA=140, max_RotB=10, max_LogP=5,
    max_MWT=500, max_HBD=5, max_HBA=10
)

# Filter outputs: {Stage}_{N}cmpds.csv (row count in filename)
df_imidazolones_druglike, _ = load_or_filter(
    lambda: filter_Veber(
        df_imidazolones_raw,
        recompute_descriptors=True,
        max_price_mol=MAX_PRICE_MOL,
        **VEBER_PRODUCTS,
    ),
    accepted_csv=stage_path("Imidazolones", filter_mode="veber"),
    rejected_csv=rejected_path("Imidazolones", "veber"),
    force_recompute=FORCE_RECOMPUTE,
    input_row_count=len(df_imidazolones_raw),
    params={"max_price_mol": MAX_PRICE_MOL, "veber": VEBER_PRODUCTS, "recompute_descriptors": True},
)

df_thiazolones_druglike, _ = load_or_filter(
    lambda: filter_Veber(
        df_thiazolones_raw,
        recompute_descriptors=True,
        max_price_mol=MAX_PRICE_MOL,
        **VEBER_PRODUCTS,
    ),
    accepted_csv=stage_path("Thiazolones", filter_mode="veber"),
    rejected_csv=rejected_path("Thiazolones", "veber"),
    force_recompute=FORCE_RECOMPUTE,
    input_row_count=len(df_thiazolones_raw),
    params={"max_price_mol": MAX_PRICE_MOL, "veber": VEBER_PRODUCTS, "recompute_descriptors": True},
)

report_df_size(df_imidazolones_druglike, "Imidazolones - Druglike")
report_df_size(df_thiazolones_druglike, "Thiazolones - Druglike")

[load_or_filter] No cache pair matches input row count (3,970,334); recomputing
[load_or_filter] Computing Imidazolones_veber.csv...
[Veber] Processing 3,970,334 molecules in 398 chunks...
[Veber] Processing chunk 100/398...
[Veber] Processing chunk 200/398...
[Veber] Processing chunk 300/398...
[filter_Veber] 2,975,383/3,970,334 accepted (74.9%), 994,951 rejected
[load_or_filter] Saved Imidazolones_veber_2975383cmpds.csv (2,975,383 accepted) + Imidazolones_rejected_veber_994951cmpds.csv (994,951 rejected)
[load_or_filter] No cache pair matches input row count (61,299); recomputing
[load_or_filter] Computing Thiazolones_veber.csv...
[Veber] Processing 61,299 molecules in 7 chunks...
[filter_Veber] 48,452/61,299 accepted (79.0%), 12,847 rejected
[load_or_filter] Saved Thiazolones_veber_48452cmpds.csv (48,452 accepted) + Thiazolones_rejected_veber_12847cmpds.csv (12,847 rejected)
[Imidazolones - Druglike] 2975383 rows
[Thiazolones - Druglike] 48452 rows


## 🔸 9. Brenk + PAINS Filter

Remove compounds containing Brenk structural alerts or PAINS substructures from products.

In [10]:
df_imidazolones_untoxic, _ = load_or_filter(
    lambda: filter_BrenkPAINS(df_imidazolones_druglike),
    accepted_csv=stage_path("Imidazolones", filter_mode="brenkpains"),
    rejected_csv=rejected_path("Imidazolones", "brenkpains"),
    force_recompute=FORCE_RECOMPUTE,
    input_row_count=len(df_imidazolones_druglike),
    params={"filter": "brenkpains", "max_price_mol": MAX_PRICE_MOL},
)

df_thiazolones_untoxic, _ = load_or_filter(
    lambda: filter_BrenkPAINS(df_thiazolones_druglike),
    accepted_csv=stage_path("Thiazolones", filter_mode="brenkpains"),
    rejected_csv=rejected_path("Thiazolones", "brenkpains"),
    force_recompute=FORCE_RECOMPUTE,
    input_row_count=len(df_thiazolones_druglike),
    params={"filter": "brenkpains", "max_price_mol": MAX_PRICE_MOL},
)

report_df_size(df_imidazolones_untoxic, "Imidazolones - Untoxic")
report_df_size(df_thiazolones_untoxic, "Thiazolones - Untoxic")

[load_or_filter] No cache pair matches input row count (2,975,383); recomputing
[load_or_filter] Computing Imidazolones_brenkpains.csv...
[filter_BrenkPAINS] 1815831/2975383 accepted (61.0%)
[filter_BrenkPAINS] Rejected: Brenk=1135527, PAINS=24025
[load_or_filter] Saved Imidazolones_brenkpains_1815831cmpds.csv (1,815,831 accepted) + Imidazolones_rejected_brenkpains_1159552cmpds.csv (1,159,552 rejected)
[load_or_filter] No cache pair matches input row count (48,452); recomputing
[load_or_filter] Computing Thiazolones_brenkpains.csv...
[filter_BrenkPAINS] 31565/48452 accepted (65.1%)
[filter_BrenkPAINS] Rejected: Brenk=16351, PAINS=536
[load_or_filter] Saved Thiazolones_brenkpains_31565cmpds.csv (31,565 accepted) + Thiazolones_rejected_brenkpains_16887cmpds.csv (16,887 rejected)
[Imidazolones - Untoxic] 1815831 rows
[Thiazolones - Untoxic] 31565 rows


## 📤 10. Export

Imidazolones and Thiazolones saved as **separate** CSV files.

In [11]:
PROD_COLS = ["ID", "SMILES", "PriceMol", "tPSA", "RotB", "LogP", "MW", "HBD", "HBA", "MR", "HvyAtm", "Atoms", "Rings", "CAtm", "HetAtm"]

# Final exports: {Stage}_{N}cmpds.csv (includes row count for clarity)
n_imi = len(df_imidazolones_untoxic)
n_thi = len(df_thiazolones_untoxic)

save_dataframe(df_imidazolones_untoxic[PROD_COLS], 
                  stage_path("Imidazolones", row_count=n_imi, filter_mode="brenkpains"))
save_dataframe(df_thiazolones_untoxic[PROD_COLS],
                  stage_path("Thiazolones", row_count=n_thi, filter_mode="brenkpains"))

# Also export clean copies to outputs/
save_dataframe(df_imidazolones_untoxic[PROD_COLS],
               f"01_library_mols_data/outputs/Imidazolones_{n_imi}cmpds.csv")
save_dataframe(df_thiazolones_untoxic[PROD_COLS],
               f"01_library_mols_data/outputs/Thiazolones_{n_thi}cmpds.csv")

print("\n=== Pipeline Complete ===")
print(f"Imidazolones: {n_imi:,} compounds")
print(f"Thiazolones:  {n_thi:,} compounds")



[save_dataframe] Saved Imidazolones_brenkpains_1815831cmpds.csv (1,815,831 rows)
[save_dataframe] Saved Thiazolones_brenkpains_31565cmpds.csv (31,565 rows)
[save_dataframe] Saved Imidazolones_1815831cmpds.csv (1,815,831 rows)
[save_dataframe] Saved Thiazolones_31565cmpds.csv (31,565 rows)

=== Pipeline Complete ===
Imidazolones: 1,815,831 compounds
Thiazolones:  31,565 compounds
